In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# פרמטרים נפוצים לטרנספילציה

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלו או בגרסאות חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
דף זה מתאר כמה מהפרמטרים הנפוצים יותר לטרנספילציה מקומית. פרמטרים אלו מוגדרים באמצעות ארגומנטים ל-[`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) או ל-[`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## מידת הקירוב
אפשר להשתמש במידת הקירוב כדי לציין עד כמה רוצים שהמעגל המתקבל יתאים למעגל הרצוי (הקלט). זהו מספר עשרוני בטווח (0.0 עד 1.0), כאשר 0.0 הוא קירוב מקסימלי ו-1.0 (ברירת המחדל) הוא ללא קירוב. ערכים קטנים יותר מחליפים דיוק בפלט עבור קלות ביצוע (כלומר, פחות שערים). ערך ברירת המחדל הוא 1.0.

בסינתזת יוניטרי דו-קיוביטית (המשמשת בשלבים הראשוניים של כל הרמות ובשלב האופטימיזציה עם רמת אופטימיזציה 3), ערך זה מציין את הנאמנות היעד של פירוק הפלט. כלומר, כמה שגיאה מוכנסת כאשר ייצוג מטריצה של מעגל מומר לשערים בדידים. אם מידת הקירוב נמוכה יותר (קירוב רב יותר), המעגל היוצא מהסינתזה יהיה שונה יותר מהמטריצה הקלט, אך סביר שיהיו לו גם פחות שערים (כי ניתן לפרק כל פעולה דו-קיוביטית שרירותית בצורה מושלמת עם לכל היותר שלושה שערי CX) וקל יותר להריץ אותו.

כאשר מידת הקירוב קטנה מ-1.0, עלולים להיוצר מעגלים עם שער CX אחד או שניים, מה שמוביל לפחות שגיאה מהחומרה, אך יותר שגיאה מהקירוב. מכיוון ש-CX הוא השער היקר ביותר מבחינת שגיאה, ייתכן שיהיה כדאי להפחית את מספרם על חשבון הנאמנות בסינתזה (טכניקה זו שימשה להגדלת הנפח הקוונטי על מכשירי IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

כדוגמה, ניצור `UnitaryGate` דו-קיוביטית אקראית שתסונתז בשלב הראשוני. הגדרת `approximation_degree` לפחות מ-1.0 עשויה לייצר מעגל מקורב. עלינו גם לציין את `basis_gates` כדי לאפשר לשיטת הסינתזה לדעת אילו שערים היא יכולה להשתמש בהם לסינתזה המקורבת.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

תוצאה זו היא `2` כי הקירוב דורש פחות שערי CX.

<span id="seed"></span>
## זרע מחולל המספרים האקראיים
חלקים מסוימים של הטרנספיילר הם סטוכסטיים, ולכן הרצות טרנספילציה חוזרות עשויות להחזיר תוצאות שונות. כדי לקבל תוצאה שניתן לשחזור, אפשר להגדיר את הזרע למחולל המספרים הפסאודו-אקראיים באמצעות הארגומנט `seed_transpiler`. הרצות חוזרות עם אותו זרע יחזירו את אותן התוצאות.

דוגמה:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## פריסה ראשונית
לפני הטרנספילציה, ה-Qubitים הכלולים במעגל שלך הם Qubitים וירטואליים שאינם בהכרח תואמים ל-Qubitים פיזיים ב-Backend היעד. ניתן לציין את המיפוי הראשוני של Qubitים וירטואליים ל-Qubitים פיזיים באמצעות הארגומנט `initial_layout`. שים לב שפריסת ה-Qubit הסופית עשויה להיות שונה מהפריסה הראשונית מכיוון שהטרנספיילר עשוי לסדר מחדש Qubitים באמצעות שערי swap או אמצעים אחרים.

בדוגמה הבאה, אנו בונים פריסה ראשונית ל-Backend המדומה [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) על ידי יצירת אובייקט [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). הפריסה שלנו ממפה את ה-Qubit הראשון של המעגל שלנו ל-Qubit 5 של Sherbrooke, ואת ה-Qubit השני של המעגל שלנו ל-Qubit 6 של Sherbrooke. שים לב ש-Qubitים פיזיים מיוצגים תמיד על ידי מספרים שלמים.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

בנוסף לציון אובייקט Layout, אפשר גם להעביר רשימה של מספרים שלמים, שבה האיבר ה-$i$ של הרשימה מכיל את ה-Qubit הפיזי שאליו יש למפות את ה-Qubit ה-$i$. לדוגמה:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

אפשר להשתמש בפונקציה [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) כדי ליצור תרשים של גרף המכשיר עם מידע על שגיאות ועם ה-Qubitים הפיזיים מסומנים. אפשר גם לצפות בתרשימים דומים בדף [Compute resources](https://quantum.cloud.ibm.com/computers).